In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# 1. Imports

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

Nesta etapa, importei as bibliotecas essenciais para manipulação de dados (pandas), visualização (matplotlib, seaborn) e todo o sistema do scikit-learn necessário para processamento, treinamento do modelo SVM e avaliação de métricas.


# 2. Salvando o dataframe

In [ ]:
dataframe = pd.read_csv("/kaggle/input/datasets/uciml/breast-cancer-wisconsin-data/data.csv")

Realizei a leitura do arquivo CSV contendo os dados do dataset diretamente do diretório fornecido pelo Kaggle, carregando em um DataFrame do Pandas.

# 3. Verificando os dados existentes

# 3.1 Info

In [ ]:
dataframe.info() #mostra as informacoes

# 3.2 Head

In [ ]:
dataframe.head() #mostra as 5 primeiras linhas

# 3.3 Shape

In [ ]:
dataframe.shape # a forma

# 3.4 Describe

In [ ]:
dataframe.describe() #a descricao detalhada

# 3.5 Is null

In [ ]:
dataframe.isnull().sum() #verifica os nulos

Executei uma Análise Exploratória de Dados inicial para verificar as dimensões do dataset, a estrutura das colunas, os tipos de dados, dados estatísticos descritivos e a presença de possiveis nulos

# 4. Limpando colunas desnecessarias e nulas

# 4.1 Nulos

In [ ]:
dataframe = dataframe.drop("Unnamed: 32", axis = 1) #apaga os nulos

# 4.2 Id

In [ ]:
dataframe = dataframe.drop("id", axis = 1) #apaga o id

# 4.3 Exibindo novamente para verificar

In [ ]:
dataframe.head() #exibi novamente

In [ ]:
dataframe.isnull().sum() #exibi novamente 

Limpei os dados fazendo com que os nulos e colunas que nao tivessem necessidade ficassem de fora

# 5. Graficos para visualização 

# 5.1 Grafico dispersao Raio x Media

In [ ]:
sns.scatterplot( #grafico de dispersao
    data=dataframe,
    x="radius_mean",
    y="area_mean",
    hue="diagnosis",
    palette = {"M": "red","B": "Blue"}
)

plt.title("Média do Raio X Média da Área")
plt.show()

# 5.2 Grafico de dispersao Textura x Suavidade

In [ ]:
sns.scatterplot(
    data=dataframe,
    x="texture_mean",
    y="smoothness_mean",
    hue="diagnosis",
    palette = {"M": "red","B": "blue"}
)

plt.title("Média da Textura X Média da Suavidade")
plt.show()

# 5.3 Grafico de dispersao Concavidade x Pontos Concavos

In [ ]:
sns.scatterplot(
    data=dataframe,
    x="concavity_mean",
    y="concave points_mean",
    hue="diagnosis",
    palette = {"M": "red","B": "blue"}
)

plt.title("Média da Concavidade X Média dos Pontos Concavos")
plt.show()

# 5.4 Raio x Media da area

In [ ]:
sns.scatterplot(
    data=dataframe,
    x="symmetry_mean",
    y="fractal_dimension_mean",
    hue="diagnosis",
    palette = {"M": "red","B": "blue"}
)

plt.title("Média do Raio X Média da Área")
plt.show()

Geramos gráficos de dispersão relacionando as características para visualizar a separabilidade das classes e a distribuição espacial das diferentes condicoes



# 5.5 Distribuição das variaveis

In [ ]:
dataframe[dataframe.columns].hist(figsize=(10, 6))
plt.suptitle("Distribuição das variáveis do dataset Breast Cancer Wisconsion", y=1.02)
plt.tight_layout()
plt.show()

# 6. SVM

# 6.1 Pasando dados para X(caracteristicas) e Y(alvo)


In [ ]:
x = dataframe[[
    "radius_mean",
    "area_mean",
    "perimeter_mean"
]]
y = dataframe["diagnosis"]
print("Formato de x: ", x.shape)
print("Formato de y: ", y.shape)

Separei o conjunto de dados em matriz de recursos X e Y confirmando suas dimensões.

# 6.2 Separação entre treino e teste

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y) #treino e teste dex e y
print("X_train: ", x_train.shape) #os prints do treino e teste
print("X_test: ", x_test.shape)
print("Y_train: ", y_train.shape)
print("Y_test: ", y_test.shape)
print("Distribuição em y_train: ", y_train.value_counts())
print("Distribuição em y_test: ", y_test.value_counts())

Dividi os dados em conjuntos de treinamento e teste (proporção 80/20), utilizando o parâmetro stratify=y para preservar a proporção exata de cada espécie de flor em ambos os conjuntos e random_state para reprodutibilidade.


# 6.3 Linear

In [ ]:
pipeline = Pipeline([ #kernel linear
   ("scaler", StandardScaler()),
   ("svm", SVC(kernel="linear", C = 1.0, random_state = 42))
])
pipeline.fit(x_train, y_train)
y_prever = pipeline.predict(x_test)
precisao = accuracy_score(y_test, y_prever)
print(f"A precisão do modelo foi: {precisao*100:.2f}%")

# 6.4 Validação cruzada

In [ ]:
validacao_cruzada = cross_val_score(pipeline, x, y, cv = 5) #validacao cruzada do linear
print("Precisão em cada verificação: ", validacao_cruzada)
print(f"A média de precisao foi: {validacao_cruzada.mean()*100:.2f}%")
print(f"O desvio padrão de precisao foi: {validacao_cruzada.std()*100:.2f}%")

# 6.5 Matriz de confusão

In [ ]:
matriz = confusion_matrix(y_test, y_prever, labels=["M", "B"]) #matriz da confusao do linear
disp = ConfusionMatrixDisplay(confusion_matrix = matriz, display_labels=["M", "B"])
disp.plot()
plt.title("Matriz de confusão")
plt.show()

Construí a matriz de confusão gráfica para visualizar detalhadamente os acertos e eventuais erros de classificação cometidos pelo modelo

# 6.6 Relatorio 

In [ ]:
print("Relatório de classificação do SVM Linear: ")  #relatorio do linear 
print(classification_report(y_test, y_prever))

# 7. RBF

In [ ]:
pipeline_rbf = Pipeline([ #kernel RBF
   ("scaler", StandardScaler()),
   ("svm", SVC(kernel="rbf", C = 1.0, gamma = "auto", random_state = 42))
])
pipeline_rbf.fit(x_train, y_train)
y_prever = pipeline_rbf.predict(x_test)
precisao = accuracy_score(y_test, y_prever)
print(f"A precisão do modelo foi: {precisao*100:.2f}%")

# 7.1 Validação Cruzada

In [ ]:
validacao_cruzada = cross_val_score(pipeline_rbf, x, y, cv = 5) #validação cruzada do RBF
print("Precisão em cada verificação: ", validacao_cruzada)
print(f"A média de precisao foi: {validacao_cruzada.mean()*100:.2f}%")
print(f"O desvio padrão de precisao foi: {validacao_cruzada.std()*100:.2f}%")

# 7.2 Matriz da Confusão

In [ ]:
matriz = confusion_matrix(y_test, y_prever, labels=["M", "B"]) #Matriz da confusao do RBF
disp = ConfusionMatrixDisplay(confusion_matrix = matriz, display_labels=["M", "B"])
disp.plot()
plt.title("Matriz de confusão")
plt.show()

# 7.3 Relatorio

In [ ]:
print("Relatório de classificação do SVM RBF: ") #Relatorio do RBF
print(classification_report(y_test, y_prever))